# Notebook 3: Carga de GloVe, Word2Vec (gensim) y Sentence-Transformers

**Proyecto Final - Inteligencia Artificial**

En este notebook preparamos los **otros tres modelos de embeddings** que vamos a comparar contra nuestra implementación de Skip-gram desde cero (Notebook 2):

1. **Word2Vec con gensim**: entrenamos word2vec sobre el mismo corpus usando la librería `gensim` (que tiene una implementación en C muy optimizada). Esto nos sirve como *sanity check*: nuestros embeddings deberían parecerse a los de gensim.
2. **GloVe preentrenado en español**: cargamos vectores GloVe entrenados sobre el Spanish Billion Words Corpus por la Universidad de Chile.
3. **Sentence-Transformers (SBERT)**: cargamos un modelo multilingüe basado en BERT para embeddings a nivel de **oración/documento** (no palabra).

## Salidas
- `data/word2vec_gensim.npy`: matriz de embeddings entrenados con gensim, alineada con nuestro vocabulario.
- `data/glove_aligned.npy`: matriz GloVe alineada con nuestro vocabulario.
- `data/glove_vocab_mask.npy`: máscara booleana indicando qué palabras de nuestro vocabulario tienen vector GloVe.
- El modelo SBERT se carga en memoria; los embeddings de documentos se generarán en el Notebook 4.




In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/proyecto_ia_final"
os.makedirs(f"{PROJECT_DIR}/data", exist_ok=True)
os.chdir(PROJECT_DIR)

print(f"Trabajando en: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Trabajando en: /content/drive/MyDrive/proyecto_ia_final


## 1. Instalación de dependencias

In [ ]:
!pip install gensim sentence-transformers -q

In [ ]:
import json
import os
import gzip
import shutil
import urllib.request
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


## 2. Cargar el corpus y vocabulario del Notebook 1

In [ ]:
corpus = np.load(DATA_DIR / "corpus_tokens.npy")

with open(DATA_DIR / "vocab.json", "r", encoding="utf-8") as f:
    vocab_data = json.load(f)

word2id = vocab_data["word2id"]
id2word = {int(k): v for k, v in vocab_data["id2word"].items()}
VOCAB_SIZE = vocab_data["vocab_size"]

print(f"Corpus:      {len(corpus):,} tokens")
print(f"Vocabulario: {VOCAB_SIZE:,} palabras")


Corpus:      25,281,989 tokens
Vocabulario: 80,000 palabras


---

## 3. Word2Vec con gensim

`gensim` es la librería estándar para entrenar word2vec en Python. Su implementación en C es entre 10x y 50x más rápida que cualquier implementación en PyTorch puro, por lo que la usamos como referencia.

Entrenamos con los **mismos hiperparámetros** que el Notebook 2 para que la comparación sea justa.

In [ ]:
from gensim.models import Word2Vec

print("Preparando corpus para gensim...")

SENTENCE_LEN = 100  # Tamaño de "oración" artificial
sentences = [
    [id2word[int(tok)] for tok in corpus[i:i+SENTENCE_LEN]]
    for i in range(0, len(corpus), SENTENCE_LEN)
]
print(f"Total de 'oraciones' para gensim: {len(sentences):,}")


Preparando corpus para gensim...
Total de 'oraciones' para gensim: 252,820


In [ ]:
# Hiperparámetros (mismos que el Notebook 2)
EMBEDDING_DIM = 100
WINDOW_SIZE = 5
NUM_NEGATIVES = 5
NUM_EPOCHS = 5
MIN_COUNT = 1  # Ya filtramos en el Notebook 1

print("Entrenando word2vec con gensim...")
gensim_model = Word2Vec(
    sentences=sentences,
    vector_size=EMBEDDING_DIM,
    window=WINDOW_SIZE,
    min_count=MIN_COUNT,
    sg=1,                  # 1 = Skip-gram (0 sería CBOW)
    negative=NUM_NEGATIVES,
    epochs=NUM_EPOCHS,
    workers=4,
    seed=42,
)

print(f"Entrenamiento completado.")
print(f"Vocabulario de gensim: {len(gensim_model.wv):,} palabras")


Entrenando word2vec con gensim...
Entrenamiento completado.
Vocabulario de gensim: 80,000 palabras


In [ ]:
# Alinear los embeddings de gensim con nuestro vocabulario
# (deberían coincidir, pero por seguridad construimos la matriz en el orden de word2id)
gensim_embeddings = np.zeros((VOCAB_SIZE, EMBEDDING_DIM), dtype=np.float32)
n_found = 0
for word, idx in word2id.items():
    if word in gensim_model.wv:
        gensim_embeddings[idx] = gensim_model.wv[word]
        n_found += 1

print(f"Palabras alineadas: {n_found:,} / {VOCAB_SIZE:,}")
np.save(DATA_DIR / "word2vec_gensim.npy", gensim_embeddings)
print(f"Guardado en data/word2vec_gensim.npy")

# Validación rápida
print("\nSanity check - palabras similares a 'rey' según gensim:")
try:
    for word, sim in gensim_model.wv.most_similar("rey", topn=5):
        print(f"  {word:20s} {sim:.4f}")
except KeyError:
    print("  'rey' no está en el vocabulario.")


Palabras alineadas: 80,000 / 80,000
Guardado en data/word2vec_gensim.npy

Sanity check - palabras similares a 'rey' según gensim:
  monarca              0.8250
  trono                0.7865
  reinar               0.7692
  regente              0.7685
  raimúndez            0.7668


---

## 4. GloVe Preentrenado en Español

Usamos los vectores GloVe entrenados por el grupo del DCC de la Universidad de Chile sobre el Spanish Billion Words Corpus (SBWC).

- **URL**: http://dcc.uchile.cl/~jperez/word-embeddings/glove-sbwc.i25.vec.gz
- **Archivo**: `glove-sbwc.i25.vec.gz` (~850 MB comprimido, ~1.6 GB descomprimido).
- **Dimensión**: 300 (lo reduciremos por SVD si queremos comparar a 100d, o lo dejamos en 300).



In [ ]:
import gzip
import shutil

GLOVE_URL = "http://dcc.uchile.cl/~jperez/word-embeddings/glove-sbwc.i25.vec.gz"
GLOVE_GZ_PATH = DATA_DIR / "glove-sbwc.i25.vec.gz"
GLOVE_PATH = DATA_DIR / "glove-sbwc.i25.vec"

if GLOVE_PATH.exists():
    print(f"{GLOVE_PATH} ya existe. Saltando descarga.")
elif GLOVE_GZ_PATH.exists():
    print(f"{GLOVE_GZ_PATH} existe. Saltando descarga, descomprimiendo...")
else:
    print(f"Descargando GloVe desde {GLOVE_URL}...")

    class DownloadProgressBar(tqdm):
        def update_to(self, b=1, bsize=1, tsize=None):
            if tsize is not None:
                self.total = tsize
            self.update(b * bsize - self.n)

    with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc="GloVe") as t:
        urllib.request.urlretrieve(GLOVE_URL, GLOVE_GZ_PATH, reporthook=t.update_to)
    print(f"Descarga completada: {GLOVE_GZ_PATH}")

# Descomprimir si todavía no se ha hecho
if not GLOVE_PATH.exists():
    print(f"Descomprimiendo {GLOVE_GZ_PATH} -> {GLOVE_PATH}...")
    with gzip.open(GLOVE_GZ_PATH, 'rb') as f_in:
        with open(GLOVE_PATH, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    print(f"Descompresión completada: {GLOVE_PATH}")


data/glove-sbwc.i25.vec ya existe. Saltando descarga.


In [ ]:
# Cargar GloVe en formato texto y alinearlo con nuestro vocabulario
print("Cargando GloVe y alineando con nuestro vocabulario...")

# Detectar dimensión leyendo la primera línea con datos
with open(GLOVE_PATH, "r", encoding="utf-8", errors="ignore") as f:
    first_line = f.readline().strip().split()
    # El archivo .vec de GloVe puede tener una cabecera (vocab_size, dim) o no
    if len(first_line) == 2 and first_line[0].isdigit():
        GLOVE_DIM = int(first_line[1])
        has_header = True
        print(f"Archivo con cabecera. Vocab: {first_line[0]}, dim: {GLOVE_DIM}")
    else:
        GLOVE_DIM = len(first_line) - 1
        has_header = False
        print(f"Archivo sin cabecera. Dim inferida: {GLOVE_DIM}")

# Construir matriz alineada
glove_embeddings = np.zeros((VOCAB_SIZE, GLOVE_DIM), dtype=np.float32)
glove_mask = np.zeros(VOCAB_SIZE, dtype=bool)

with open(GLOVE_PATH, "r", encoding="utf-8", errors="ignore") as f:
    if has_header:
        f.readline()  # saltar cabecera
    for line in tqdm(f, desc="Cargando GloVe"):
        parts = line.rstrip().split(" ")
        if len(parts) < GLOVE_DIM + 1:
            continue
        word = parts[0]
        if word in word2id:
            vec = np.array(parts[1:GLOVE_DIM+1], dtype=np.float32)
            idx = word2id[word]
            glove_embeddings[idx] = vec
            glove_mask[idx] = True

n_covered = glove_mask.sum()
print(f"\nCobertura de GloVe sobre nuestro vocabulario: {n_covered:,} / {VOCAB_SIZE:,} ({100*n_covered/VOCAB_SIZE:.1f}%)")

# Para palabras sin vector GloVe, dejamos ceros (las marcaremos para no usarlas)
np.save(DATA_DIR / "glove_aligned.npy", glove_embeddings)
np.save(DATA_DIR / "glove_vocab_mask.npy", glove_mask)
print(f"GloVe alineado guardado: {glove_embeddings.shape}")
print(f"Máscara guardada: {glove_mask.shape}")


Cargando GloVe y alineando con nuestro vocabulario...
Archivo con cabecera. Vocab: 855380, dim: 300


Cargando GloVe: 0it [00:00, ?it/s]


Cobertura de GloVe sobre nuestro vocabulario: 79,089 / 80,000 (98.9%)
GloVe alineado guardado: (80000, 300)
Máscara guardada: (80000,)


In [ ]:
# Validar GloVe
def cosine_sim(a, b):
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

def most_similar_glove(word, top_k=5):
    if word not in word2id or not glove_mask[word2id[word]]:
        return None
    idx = word2id[word]
    q = glove_embeddings[idx]
    # Normalizar todas las filas que tengan máscara
    norms = np.linalg.norm(glove_embeddings, axis=1, keepdims=True)
    normed = np.where(norms > 0, glove_embeddings / (norms + 1e-9), 0)
    sims = normed @ (q / (np.linalg.norm(q) + 1e-9))
    sims[~glove_mask] = -1  # ignorar palabras sin vector
    top = np.argsort(-sims)[:top_k+1]
    return [(id2word[int(i)], float(sims[i])) for i in top if i != idx][:top_k]

print("Validación de GloVe - similares a algunas palabras:")
for w in ["rey", "música", "guerra", "amor"]:
    print(f"\n  {w}:")
    res = most_similar_glove(w)
    if res is None:
        print(f"    (sin vector GloVe)")
    else:
        for ww, ss in res:
            print(f"    {ww:20s} {ss:.4f}")


Validación de GloVe - similares a algunas palabras:

  rey:
    monarca              0.7613
    reina                0.6679
    alfonso              0.6574
    príncipe             0.6559
    reyes                0.6464

  música:
    musical              0.7977
    musicales            0.7595
    canciones            0.6810
    pop                  0.6522
    rock                 0.6510

  guerra:
    guerras              0.7044
    militar              0.6542
    conflicto            0.6508
    armada               0.6451
    ejército             0.6388

  amor:
    pasión               0.6702
    alma                 0.6649
    eterno               0.6320
    vida                 0.6247
    enamorado            0.6146


---

## 5. Sentence-Transformers (SBERT)

Cargamos un modelo multilingüe preentrenado. A diferencia de word2vec/GloVe, SBERT genera embeddings a nivel de **oración o documento completo**, no de palabra individual. Internamente usa una arquitectura Transformer (BERT) refinada para producir vectores donde la similitud coseno aproxima la similitud semántica entre oraciones.

Modelo elegido: `paraphrase-multilingual-MiniLM-L12-v2`
- Soporta 50+ idiomas incluido español.
- Dimensión de embedding: 384.
- Tamaño: ~120 MB.
- Muy buen balance velocidad/calidad para nuestro caso.

In [ ]:
from sentence_transformers import SentenceTransformer

print("Cargando modelo SBERT multilingüe...")
sbert_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Mover a GPU si está disponible
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
sbert_model = sbert_model.to(device)
print(f"Modelo cargado en {device}.")
print(f"Dimensión de embedding: {sbert_model.get_sentence_embedding_dimension()}")


Cargando modelo SBERT multilingüe...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo cargado en cpu.
Dimensión de embedding: 384


/tmp/ipykernel_22170/2154736556.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Dimensión de embedding: {sbert_model.get_sentence_embedding_dimension()}")


In [ ]:
# Prueba rápida: codificar un par de oraciones y ver su similitud
test_sentences = [
    "El gato duerme en el sofá.",
    "Un felino descansa sobre el mueble.",
    "Los aviones vuelan muy alto.",
]

test_embeddings = sbert_model.encode(test_sentences, convert_to_numpy=True)
print(f"Shape de embeddings: {test_embeddings.shape}")

# Similitud entre oraciones
from numpy.linalg import norm
def cos(a, b):
    return (a @ b) / (norm(a) * norm(b))

print(f"\nSimilitud entre oraciones:")
print(f"  '{test_sentences[0]}'")
print(f"  '{test_sentences[1]}'")
print(f"  -> cos = {cos(test_embeddings[0], test_embeddings[1]):.4f}  (deberían ser similares)")
print()
print(f"  '{test_sentences[0]}'")
print(f"  '{test_sentences[2]}'")
print(f"  -> cos = {cos(test_embeddings[0], test_embeddings[2]):.4f}  (deberían ser distintas)")


Shape de embeddings: (3, 384)

Similitud entre oraciones:
  'El gato duerme en el sofá.'
  'Un felino descansa sobre el mueble.'
  -> cos = 0.6679  (deberían ser similares)

  'El gato duerme en el sofá.'
  'Los aviones vuelan muy alto.'
  -> cos = -0.0302  (deberían ser distintas)


In [ ]:
# Verificar que todo quedó bien guardado
import os

files_to_check = [
    "word2vec_pytorch.npy",
    "word2vec_gensim.npy",
    "glove_aligned.npy",
    "glove_vocab_mask.npy",
]

print("Archivos en data/:")
for fname in files_to_check:
    path = DATA_DIR / fname
    if path.exists():
        size_mb = path.stat().st_size / 1e6
        print(f"  ✓ {fname:30s} ({size_mb:.1f} MB)")
    else:
        print(f"  ✗ {fname:30s} (FALTA)")

print("\n¡Notebook 3 completado!")


Archivos en data/:
  ✓ word2vec_pytorch.npy           (32.0 MB)
  ✓ word2vec_gensim.npy            (32.0 MB)
  ✓ glove_aligned.npy              (96.0 MB)
  ✓ glove_vocab_mask.npy           (0.1 MB)

¡Notebook 3 completado!
